### Task 1: Comparative Descriptive Statistics
The goal of this task is to determine the "Duration Gap" between our two user segments. By calculating the mean, median, and extreme values for ride lengths, we can begin to identify the different ways Members and Casual riders utilize the bike-share service. We convert seconds to minutes to make the findings more actionable for stakeholders.

In [8]:
import pandas as pd
import numpy as np

# 1. Load the corrected dataset
df = pd.read_csv('../data_processed/cyclistic_final_clean.csv')

# 2. Re-apply the categorical order for sorting
# Note: This ensures that when we eventually group by day, it sorts Sunday-Saturday
day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
df['day_of_week'] = pd.Categorical(df['day_of_week'], categories=day_order, ordered=True)

# 3. Calculate summary statistics (Mean, Median, Max, Min)
# No extra division needed—the data is already in minutes!
duration_summary = df.groupby('member_casual')['ride_length'].agg(['mean', 'median', 'max', 'min'])

# 4. Rename columns for a professional report
duration_summary.columns = ['Avg Trip (Min)', 'Median Trip (Min)', 'Max Trip (Min)', 'Min Trip (Min)']

print("--- Trip Duration Summary (Minutes) ---")
print(duration_summary.round(2))

--- Trip Duration Summary (Minutes) ---
               Avg Trip (Min)  Median Trip (Min)  Max Trip (Min)  \
member_casual                                                      
casual                  19.97              11.89         1499.97   
member                  12.27               8.76         1499.97   

               Min Trip (Min)  
member_casual                  
casual                    1.0  
member                    1.0  


### Task 2: Average Ride Length by Day of the Week
We are now breaking down the average trip duration by day of the week. This will help us identify if behavior shifts during the weekend versus the work week for both user segments.

In [9]:
# Calculate the average ride length for each day of the week for each user type
day_of_week_analysis = df.groupby(['member_casual', 'day_of_week'], observed=False)['ride_length'].mean().round(2)

print("--- Average Ride Length by Day (Minutes) ---")
print(day_of_week_analysis)

--- Average Ride Length by Day (Minutes) ---
member_casual  day_of_week
casual         Sunday         23.23
               Monday         19.82
               Tuesday        17.56
               Wednesday      16.47
               Thursday       17.53
               Friday         19.72
               Saturday       22.40
member         Sunday         13.51
               Monday         11.90
               Tuesday        11.91
               Wednesday      11.73
               Thursday       11.81
               Friday         12.25
               Saturday       13.37
Name: ride_length, dtype: float64


### Task 3: Ridership Volume by Day of the Week
While Casual riders take longer trips, we need to see which group is actually using the bikes more frequently. This task calculates the total number of rides per day for each user type.

In [10]:
# Calculate the number of rides for each day of the week for each user type
ridership_volume = df.groupby(['member_casual', 'day_of_week'], observed=False)['ride_id'].count()

print("--- Total Number of Rides by Day ---")
print(ridership_volume)

--- Total Number of Rides by Day ---
member_casual  day_of_week
casual         Sunday         319066
               Monday         221367
               Tuesday        219379
               Wednesday      214390
               Thursday       248726
               Friday         309108
               Saturday       397315
member         Sunday         374950
               Monday         496735
               Tuesday        563625
               Wednesday      549105
               Thursday       567363
               Friday         523258
               Saturday       441842
Name: ride_id, dtype: int64


### Task 4: Monthly Ridership Trends
To understand the impact of seasonality, we are analyzing the number of rides per month. This will help us identify the peak season for casual riders, which is the ideal time for marketing campaigns aimed at converting them to annual members.

In [11]:
# First, ensure 'started_at' is datetime and extract the month
df['month'] = pd.to_datetime(df['started_at']).dt.month_name()

# Set chronological order for months
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']
df['month'] = pd.Categorical(df['month'], categories=month_order, ordered=True)

# Calculate volume by month
monthly_volume = df.groupby(['member_casual', 'month'], observed=False)['ride_id'].count()

print("--- Total Number of Rides by Month ---")
print(monthly_volume)

--- Total Number of Rides by Month ---
member_casual  month    
casual         January       23876
               February      40076
               March         82870
               April        105271
               May          175667
               June         278699
               July         308443
               August       323545
               September    254728
               October      214390
               November      94698
               December      27088
member         January      109862
               February     157237
               March        208462
               April        257922
               May          313997
               June         379519
               July         430404
               August       443127
               September    440952
               October      414099
               November     251923
               December     109374
Name: ride_id, dtype: int64


### Task 5: Top 10 Start Stations by User Type
To identify the best physical locations for marketing advertisements, we are finding the most popular starting stations for both Members and Casual riders. This will highlight the geographic divide between tourist/leisure spots and residential/commuter hubs.

In [12]:
# Top 10 Start Stations for Casual Riders
top_10_casual = df[df['member_casual'] == 'casual']['start_station_name'].value_counts().head(10)

# Top 10 Start Stations for Members
top_10_member = df[df['member_casual'] == 'member']['start_station_name'].value_counts().head(10)

print("--- Top 10 Stations for Casual Riders ---")
print(top_10_casual)
print("\n--- Top 10 Stations for Members ---")
print(top_10_member)

--- Top 10 Stations for Casual Riders ---
start_station_name
Unknown                               381099
DuSable Lake Shore Dr & Monroe St      30874
Navy Pier                              27597
Streeter Dr & Grand Ave                22919
Michigan Ave & Oak St                  22137
DuSable Lake Shore Dr & North Blvd     19149
Millennium Park                        18796
Shedd Aquarium                         16604
Theater on the Lake                    15589
Dusable Harbor                         15394
Name: count, dtype: int64

--- Top 10 Stations for Members ---
start_station_name
Unknown                         720303
Kingsbury St & Kinzie St         27868
Clinton St & Washington Blvd     24457
Canal St & Madison St            22103
Clinton St & Madison St          21340
Clark St & Elm St                19889
State St & Chicago Ave           19001
Wells St & Elm St                18713
Clinton St & Jackson Blvd        18270
Wells St & Concord Ln            18237
Name: count, dtyp

### Task 6: Data Aggregation for Visualization (Tableau Export)
To ensure optimal performance in the "Share" phase, we are exporting four targeted summary tables. While our full dataset contains over 5.4 million records, aggregating specific metrics (weekly averages, monthly volume, and station hotspots) allows Tableau to render interactive dashboards significantly faster. This approach follows the data engineering principle of 'reducing data at the source' before the reporting layer.

In [13]:
# 1. Weekly Summary: Comparing volume and duration by day of week
weekly_viz = df.groupby(['member_casual', 'day_of_week'], observed=False).agg(
    total_rides=('ride_id', 'count'),
    avg_duration_mins=('ride_length', 'mean')
).reset_index()
weekly_viz.to_csv('../data_processed/viz_weekly_summary.csv', index=False)

# 2. Monthly Summary: Identifying seasonal surges
monthly_viz = df.groupby(['member_casual', 'month'], observed=False).agg(
    total_rides=('ride_id', 'count')
).reset_index()
monthly_viz.to_csv('../data_processed/viz_monthly_summary.csv', index=False)

# 3. Station Summary: Mapping geographic hotspots (Top 100 stations)
station_viz = df.groupby(['member_casual', 'start_station_name', 'start_lat', 'start_lng']).agg(
    total_rides=('ride_id', 'count')
).reset_index()
station_viz = station_viz.sort_values('total_rides', ascending=False).head(200)
station_viz.to_csv('../data_processed/viz_station_summary.csv', index=False)

# 4. Bike Type Summary: Understanding equipment preference
bike_type_viz = df.groupby(['member_casual', 'rideable_type']).agg(
    total_rides=('ride_id', 'count')
).reset_index()
bike_type_viz.to_csv('../data_processed/viz_bike_type_summary.csv', index=False)

print("✅ Four summary CSVs have been exported to the 'data_processed' folder for Tableau.")

✅ Four summary CSVs have been exported to the 'data_processed' folder for Tableau.


**Key Outcomes:**
* **Performance:** The resulting files are lightweight, allowing for instant filtering in Tableau Public.
* **Geospatial Integrity:** Station coordinates (Lat/Long) are preserved to enable high-resolution mapping.
* **Granularity:** We maintained the 'member_casual' split across all files to ensure our comparative analysis remains the focal point of the final story.